In [3]:
## 실험: 커널 크기 비교 — 3×3 vs 5×5 vs 7×7

# 커널 크기가 CNN 성능에 어떤 영향을 미치는지 직접 실험해봅니다.

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 데이터 로드 (CIFAR-10) ──────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

# ── 커널 크기별 CNN 정의 ──────────────────────────────────────────
class KernelCNN(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        pad = kernel_size // 2   # same-padding 유지
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size, padding=pad), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size, padding=pad), nn.ReLU(),
            nn.MaxPool2d(2),                             # 32→16
            nn.Conv2d(64, 128, kernel_size, padding=pad), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16→8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_and_eval(kernel_size, epochs=10):
    model = KernelCNN(kernel_size).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    params = count_params(model)
    print(f"\n[kernel={kernel_size}x{kernel_size}] 파라미터 수: {params:,}")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
        train_acc = 100 * correct / total

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                val_correct += model(imgs).argmax(1).eq(labels).sum().item()
                val_total   += labels.size(0)
        test_acc = 100 * val_correct / val_total
        print(f"  Epoch {epoch}/{epochs} — train_acc: {train_acc:.1f}%, test_acc: {test_acc:.1f}%")

    return test_acc, params

# ── 실험 실행 ───────────────────────────────────────────────────
results = {}
for ks in [3, 5, 7, 9]:
    acc, params = train_and_eval(ks, epochs=10)
    results[ks] = (acc, params)

print("\n" + "="*50)
print("최종 결과 요약")
print("="*50)
print(f"{'커널':>8} {'파라미터':>12} {'Test Acc':>10}")
for ks, (acc, params) in results.items():
    print(f"{ks}x{ks}:   {params:>12,}   {acc:>8.1f}%")


# **관찰:**
# - 3×3 커널은 파라미터 수가 가장 적으면서도 정확도가 높은 경우가 많음
# - 7×7 커널은 파라미터 수가 크게 증가하지만 32×32 CIFAR-10에서는 오히려 성능이 낮을 수 있음
# - 같은 수용 영역을 위해 3×3을 2번 쌓는 것이 7×7 1번보다 효율적인 이유가 여기 있음

# **해석:**
# - 작은 이미지(32×32)에서 큰 커널은 지나치게 큰 영역을 한 번에 보려 함
# - 3×3 스택은 더 많은 비선형 변환을 학습할 수 있어 표현력이 높음
# - VGGNet이 3×3만 사용하는 전략의 근거가 바로 이 실험 결과와 같은 관찰에서 나옴

Using device: cuda

[kernel=3x3] 파라미터 수: 2,193,226
  Epoch 1/10 — train_acc: 52.3%, test_acc: 66.5%
  Epoch 2/10 — train_acc: 68.8%, test_acc: 72.1%
  Epoch 3/10 — train_acc: 74.9%, test_acc: 74.8%
  Epoch 4/10 — train_acc: 79.6%, test_acc: 75.6%
  Epoch 5/10 — train_acc: 83.5%, test_acc: 76.0%
  Epoch 6/10 — train_acc: 87.0%, test_acc: 76.9%
  Epoch 7/10 — train_acc: 89.9%, test_acc: 76.5%
  Epoch 8/10 — train_acc: 91.8%, test_acc: 76.5%
  Epoch 9/10 — train_acc: 93.8%, test_acc: 76.8%
  Epoch 10/10 — train_acc: 94.4%, test_acc: 76.1%

[kernel=5x5] 파라미터 수: 2,358,602
  Epoch 1/10 — train_acc: 48.4%, test_acc: 60.5%
  Epoch 2/10 — train_acc: 66.2%, test_acc: 69.5%
  Epoch 3/10 — train_acc: 73.5%, test_acc: 72.5%
  Epoch 4/10 — train_acc: 78.3%, test_acc: 75.3%
  Epoch 5/10 — train_acc: 82.0%, test_acc: 75.8%
  Epoch 6/10 — train_acc: 85.7%, test_acc: 75.1%
  Epoch 7/10 — train_acc: 88.3%, test_acc: 75.9%
  Epoch 8/10 — train_acc: 90.4%, test_acc: 75.3%
  Epoch 9/10 — train_acc: 92.0%, t

In [ ]:
## 실험: Dropout 비율 — p=0, 0.25, 0.5, 0.75

# Dropout 비율에 따른 과적합 방지 효과를 훈련/테스트 정확도 갭으로 측정합니다.

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 데이터 로드 (CIFAR-10) ──────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

# ── Dropout 가변 CNN ────────────────────────────────────────────
class DropoutCNN(nn.Module):
    def __init__(self, drop_p):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512), nn.ReLU(),
            nn.Dropout(p=drop_p),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(p=drop_p),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            correct += model(imgs).argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return 100 * correct / total

def run_dropout_experiment(drop_p, epochs=10):
    model = DropoutCNN(drop_p).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    print(f"\n[Dropout p={drop_p}]")
    for epoch in range(1, epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    train_acc = evaluate(model, train_loader)
    test_acc  = evaluate(model, test_loader)
    gap = train_acc - test_acc
    print(f"  Train Acc: {train_acc:.1f}%  |  Test Acc: {test_acc:.1f}%  |  Gap: {gap:.1f}%")
    return train_acc, test_acc

# ── 실험 실행 ───────────────────────────────────────────────────
dropout_rates = [0.0, 0.25, 0.5, 0.75]
results = {}
for p in dropout_rates:
    train_acc, test_acc = run_dropout_experiment(p, epochs=10)
    results[p] = (train_acc, test_acc)

print("\n" + "="*60)
print("최종 결과 요약")
print("="*60)
print(f"{'Dropout p':>10} {'Train Acc':>12} {'Test Acc':>10} {'Gap':>8}")
for p, (tr, te) in results.items():
    print(f"{p:>10.2f}   {tr:>10.1f}%   {te:>8.1f}%   {tr-te:>6.1f}%")


# **관찰:**
# - p=0 (Dropout 없음): Train Acc가 가장 높고 Gap도 가장 큼 → 과적합 발생
# - p=0.25 ~ 0.5: Test Acc가 최적 구간, Gap이 줄어듦
# - p=0.75: Train/Test 모두 낮아짐 → 과소적합(underfitting) 발생

# **해석:**
# - Dropout은 훈련 시 랜덤하게 뉴런을 끄므로 모델이 특정 경로에 의존하지 못하게 함
# - 적절한 Dropout(0.25~0.5)은 훈련-테스트 갭을 줄여 일반화 성능을 높임
# - 너무 높은 Dropout(0.75)은 모델의 학습 능력 자체를 해침
# - 실제 프로젝트에서는 0.3~0.5 범위를 먼저 시도하는 것이 일반적

In [ ]:
## 실험: 깊이별 성능 — 2, 4, 6, 8 Conv Layers

# 컨볼루션 레이어 수(깊이)에 따른 정확도와 훈련 속도 트레이드오프를 측정합니다.

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 데이터 로드 (CIFAR-10) ──────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

# ── 깊이 가변 CNN ───────────────────────────────────────────────
def build_deep_cnn(num_conv_layers):
    """
    num_conv_layers: 2, 4, 6, 8 중 하나
    - MaxPool은 2번만 사용 (32→16→8) 유지
    - 추가 레이어는 동일 해상도에서 쌓음
    """
    channels = [3, 32, 64, 64, 128, 128, 256, 256, 256]
    layers = []
    for i in range(num_conv_layers):
        in_c  = channels[i]
        out_c = channels[i + 1]
        layers.append(nn.Conv2d(in_c, out_c, 3, padding=1))
        layers.append(nn.BatchNorm2d(out_c))
        layers.append(nn.ReLU())
        if i == 1:
            layers.append(nn.MaxPool2d(2))  # 32→16
        if i == 3:
            layers.append(nn.MaxPool2d(2))  # 16→8

    # spatial 크기 계산
    spatial = 32
    if num_conv_layers >= 2: spatial = 16
    if num_conv_layers >= 4: spatial = 8
    final_channels = channels[num_conv_layers]

    class DeepCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(*layers)
            self.pool = nn.AdaptiveAvgPool2d((4, 4))
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(final_channels * 4 * 4, 256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, 10)
            )
        def forward(self, x):
            return self.classifier(self.pool(self.features(x)))

    return DeepCNN()

def run_depth_experiment(num_layers, epochs=10):
    model = build_deep_cnn(num_layers).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[Conv Layers={num_layers}] 파라미터: {params:,}")

    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        correct, total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            correct += out.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
        print(f"  Epoch {epoch}: train_acc={100*correct/total:.1f}%")
    elapsed = time.time() - start

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            val_correct += model(imgs).argmax(1).eq(labels).sum().item()
            val_total   += labels.size(0)
    test_acc = 100 * val_correct / val_total
    print(f"  → Test Acc: {test_acc:.1f}%  |  Training Time: {elapsed:.1f}s")
    return test_acc, elapsed

# ── 실험 실행 ───────────────────────────────────────────────────
results = {}
for depth in [2, 4, 6, 8]:
    acc, t = run_depth_experiment(depth, epochs=10)
    results[depth] = (acc, t)

print("\n" + "="*60)
print("최종 결과 요약")
print("="*60)
print(f"{'Depth':>8} {'Test Acc':>10} {'Time(s)':>10}")
for depth, (acc, t) in results.items():
    print(f"{depth:>8}   {acc:>8.1f}%   {t:>8.1f}s")


# **관찰:**
# - 2 레이어: 가장 빠르지만 정확도가 낮음 (특징 추출 능력 부족)
# - 4~6 레이어: 정확도와 속도의 균형점
# - 8 레이어: CIFAR-10의 작은 이미지(32×32)에서는 과도한 깊이로 성능 향상이 미미하거나 오히려 감소

# **해석:**
# - 깊이가 깊을수록 더 복잡한 특징을 학습 가능 (계층적 표현)
# - 하지만 입력 해상도가 낮으면 너무 많은 Conv 레이어가 공간 정보를 과도하게 압축
# - Batch Normalization을 사용하면 더 깊은 네트워크도 안정적으로 훈련 가능
# - ImageNet처럼 고해상도 이미지에서는 더 깊은 네트워크(ResNet-50, 101)가 빛을 발함

In [4]:
## 실험: 학습률 민감도 — lr=0.1, 0.01, 0.001, 0.0001

# 학습률이 수렴/발산에 미치는 영향을 epoch별 loss 추이로 관찰합니다.


import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 데이터 로드 (CIFAR-10) ──────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,
                                          download=True, transform=transform)
test_set  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                          download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=2)

# ── 기준 CNN (모든 실험에서 동일 아키텍처) ─────────────────────
class BaseCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def run_lr_experiment(lr, epochs=10):
    torch.manual_seed(42)   # 동일 초기화
    model = BaseCNN().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    print(f"\n[lr={lr}]")
    epoch_losses = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)
        avg_loss  = total_loss / total
        train_acc = 100 * correct / total

        # NaN 체크
        if torch.isnan(torch.tensor(avg_loss)):
            print(f"  Epoch {epoch}: Loss=NaN (발산!)")
            epoch_losses.append(float('nan'))
        else:
            epoch_losses.append(avg_loss)
            print(f"  Epoch {epoch}: loss={avg_loss:.4f}, train_acc={train_acc:.1f}%")

    # 최종 테스트 정확도
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            val_correct += model(imgs).argmax(1).eq(labels).sum().item()
            val_total   += labels.size(0)
    test_acc = 100 * val_correct / val_total
    print(f"  → 최종 Test Acc: {test_acc:.1f}%")
    return epoch_losses, test_acc

# ── 실험 실행 ───────────────────────────────────────────────────
learning_rates = [0.1, 0.01, 0.001, 0.0001]
all_results = {}
for lr in learning_rates:
    losses, acc = run_lr_experiment(lr, epochs=10)
    all_results[lr] = (losses, acc)

print("\n" + "="*60)
print("최종 결과 요약")
print("="*60)
print(f"{'lr':>8} {'E1 Loss':>10} {'E8 Loss':>10} {'Test Acc':>10}")
for lr, (losses, acc) in all_results.items():
    l1 = f"{losses[0]:.4f}" if losses[0] == losses[0] else "NaN"
    l8 = f"{losses[-1]:.4f}" if losses[-1] == losses[-1] else "NaN"
    print(f"{lr:>8}   {l1:>10}   {l8:>10}   {acc:>8.1f}%")

print("\n학습률별 특징:")
print("  lr=0.1   → AdamW에서도 불안정, loss가 oscillate하거나 발산")
print("  lr=0.01  → 빠르게 수렴하지만 최적점을 놓칠 수 있음")
print("  lr=0.001 → AdamW의 기본값, 안정적 수렴, 좋은 성능")
print("  lr=0.0001→ 너무 느린 수렴, 5에폭에서 아직 학습 초반")


# **관찰:**
# - lr=0.1: Loss가 초반에 높거나 불안정하게 진동 (AdamW도 흔들림)
# - lr=0.01: 수렴하지만 lr=0.001보다 최종 성능이 낮을 수 있음
# - lr=0.001: AdamW의 기본값, 가장 안정적인 수렴 곡선
# - lr=0.0001: loss가 천천히 감소, 8에폭에서도 수렴 미완

# **해석:**
# - 학습률이 너무 크면 gradient update가 최적점을 뛰어넘어 발산
# - 학습률이 너무 작으면 수렴 속도가 극도로 느려짐
# - AdamW는 적응형 학습률이라 SGD보다 lr에 덜 민감하지만, 그래도 lr=0.001이 황금값
# - 실전에서는 lr=1e-3로 시작하고, 학습 곡선이 plateau에 도달하면 lr scheduler로 감소시킴

Using device: cuda

[lr=0.1]
  Epoch 1: loss=18.4264, train_acc=12.2%
  Epoch 2: loss=2.2623, train_acc=14.0%
  Epoch 3: loss=2.2350, train_acc=14.9%
  Epoch 4: loss=2.2194, train_acc=15.4%
  Epoch 5: loss=2.2092, train_acc=16.1%
  Epoch 6: loss=2.2113, train_acc=15.7%
  Epoch 7: loss=2.1799, train_acc=15.7%
  Epoch 8: loss=2.1163, train_acc=16.4%
  Epoch 9: loss=2.0852, train_acc=16.7%
  Epoch 10: loss=2.0674, train_acc=17.1%
  → 최종 Test Acc: 19.5%

[lr=0.01]
  Epoch 1: loss=2.8867, train_acc=10.1%
  Epoch 2: loss=2.2946, train_acc=10.5%
  Epoch 3: loss=2.0682, train_acc=19.3%
  Epoch 4: loss=1.8529, train_acc=26.3%
  Epoch 5: loss=1.7607, train_acc=29.5%
  Epoch 6: loss=1.7118, train_acc=31.2%
  Epoch 7: loss=1.6814, train_acc=32.6%
  Epoch 8: loss=1.6331, train_acc=35.7%
  Epoch 9: loss=1.6033, train_acc=37.6%
  Epoch 10: loss=1.5747, train_acc=38.7%
  → 최종 Test Acc: 47.5%

[lr=0.001]
  Epoch 1: loss=1.4749, train_acc=47.8%
  Epoch 2: loss=1.0248, train_acc=63.6%
  Epoch 3: loss=0.8